# BERT Fine-Tuning — Updated Colab Notebook

This notebook is an updated, teaching-focused version of the older `BERT_Finetuning.ipynb`.

We will learn three common BERT fine-tuning tasks:

1. **Sequence Classification** — IMDb sentiment analysis
2. **Token Classification (NER)** — WikiAnn named-entity recognition
3. **Extractive Question Answering** — SQuAD

The notebook uses the modern Hugging Face `Auto*` APIs, dynamic padding, `Trainer`, and current `TrainingArguments` naming.

> **Recommended:** Run the sections one at a time rather than blindly using **Run all**. The three tasks are independent.

## 0. What changed from the old notebook?

The original notebook contains several older patterns:

- `!pip install --upgrade datasets fsspec transformers` — this can create Colab dependency conflicts.
- `BertTokenizer` / `BertFor...` are replaced below with `AutoTokenizer` / `AutoModelFor...` where appropriate.
- Fixed `padding="max_length"` is replaced by **dynamic padding** for classification and NER.
- The current `Trainer` API uses `processing_class=tokenizer`.
- Current `TrainingArguments` uses `eval_strategy` rather than the older `evaluation_strategy`.
- Hugging Face authentication uses `hf auth login` or the notebook login helper.
- The notebook's custom PyTorch Dataset classes are retained conceptually in the explanations, but the main examples use Hugging Face `Dataset.map()` + `Trainer` because that is simpler for standard fine-tuning.

In [ ]:
# CELL 1 — Install only what we actually need.
# We deliberately DO NOT upgrade fsspec here.
# Colab already provides many of these packages and changing fsspec can
# conflict with packages such as gcsfs.

!pip install -q -U huggingface_hub

In [ ]:
# CELL 2 — Check the environment before doing any training.

import torch
import transformers
import datasets
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Concept: What is fine-tuning?

A pretrained BERT model has already learned general language representations from large amounts of text.

**Fine-tuning** means:

```text
Pretrained BERT
      ↓
Add a task-specific head
      ↓
Train on our labeled dataset
      ↓
Updated model
      ↓
Prediction for our task
```

For sentiment classification:

```text
Review → BERT → Classification Head → Positive / Negative
```

The important distinction is:

- **Pretraining:** learn general language representations.
- **Fine-tuning:** adapt those representations to a specific supervised task.

# PART 1 — BERT for Text Classification

We will fine-tune BERT on the IMDb dataset.

The dataset contains:

- `text` → movie review
- `label` → 0 or 1

```text
0 = Negative
1 = Positive
```

You do **not** need a `data.zip` file for this example. `load_dataset("imdb")` downloads the dataset from the Hugging Face Hub.

In [ ]:
# CELL 3 — Imports for text classification

import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# CELL 4 — Load IMDb

dataset = load_dataset("imdb")

print(dataset)
print("\nTraining columns:", dataset["train"].column_names)
print("\nExample:")
print(dataset["train"][0])

## Concept: Dataset structure

A Hugging Face `DatasetDict` normally contains splits such as:

```text
train
test
validation
```

For IMDb:

```text
train → used to update model weights
test  → used to measure performance
```

The `label` is the target that the model learns to predict.

In [ ]:
# CELL 5 — Create a smaller reproducible subset for Colab practice.
# The full IMDb dataset is larger. Start small while learning.

train_dataset = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(2000))
)

test_dataset = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(500))
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# CELL 6 — Inspect the raw data before preprocessing.

print("TEXT:")
print(train_dataset[0]["text"][:500])

print("\nLABEL:")
print(train_dataset[0]["label"])

## Concept: Tokenization

BERT does not consume raw Python strings directly.

Example:

```text
"This movie is amazing"
        ↓
Tokenizer
        ↓
[CLS] this movie is amazing [SEP]
        ↓
input_ids + attention_mask
```

- `input_ids` = integer IDs representing tokens.
- `attention_mask` = 1 for real tokens and 0 for padding.
- `truncation=True` prevents very long reviews from exceeding the chosen maximum length.

We avoid padding every example to 256 tokens during preprocessing. Instead, we use **dynamic padding** when batches are created.

In [ ]:
# CELL 7 — Load the tokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

In [ ]:
# CELL 8 — Test tokenization on one sentence

sample_text = "This movie was amazing and I loved the acting."

encoded = tokenizer(
    sample_text,
    truncation=True,
    max_length=256
)

print(encoded)
print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [ ]:
# CELL 9 — Tokenize the complete dataset.
# No padding here — dynamic padding will happen later at batch creation time.

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train[0].keys())

## Concept: `map(..., batched=True)`

Instead of tokenizing one row at a time:

```text
row 1 → tokenizer
row 2 → tokenizer
row 3 → tokenizer
```

`batched=True` sends a batch of examples to the preprocessing function.

This is generally faster and is the standard Hugging Face preprocessing pattern.

In [ ]:
# CELL 10 — Remove raw text and rename the target column.

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

print(tokenized_train[0].keys())

## Concept: Why call the target `labels`?

Hugging Face Transformer models conventionally accept the target through a `labels` argument.

For sequence classification:

```python
model(input_ids=..., attention_mask=..., labels=...)
```

The model can then calculate the classification loss automatically.

In [ ]:
# CELL 11 — Dynamic padding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Dynamic padding collator created.")

## Concept: Dynamic padding

Suppose one batch contains:

```text
Review A → 40 tokens
Review B → 75 tokens
Review C → 60 tokens
```

Dynamic padding makes the batch:

```text
40 → 75
75 → 75
60 → 75
```

instead of padding all examples to 256.

This reduces unnecessary computation and memory usage. Hugging Face recommends dynamic padding for this kind of task. citeturn0search0turn0search2

In [ ]:
# CELL 12 — Load pretrained BERT with a classification head.

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Give the model human-readable label names.
model.config.id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

model.config.label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

print(model.config.id2label)

## Concept: What was added to BERT?

`bert-base-uncased` is the pretrained encoder.

`AutoModelForSequenceClassification` adds a task-specific classification head.

Conceptually:

```text
Text
 ↓
Tokenizer
 ↓
BERT Encoder
 ↓
[CLS] representation
 ↓
Classification Layer
 ↓
2 logits
 ↓
Negative / Positive
```

The pretrained BERT parameters are also updated during normal full fine-tuning.

In [ ]:
# CELL 13 — Evaluation metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }

## Concept: Accuracy vs F1

**Accuracy**

```text
correct predictions / total predictions
```

**F1**

F1 combines precision and recall:

```text
F1 = 2 × precision × recall / (precision + recall)
```

For balanced binary sentiment data, accuracy is easy to understand. F1 is useful when class distribution is uneven or when you want a metric that considers both precision and recall.

In [ ]:
# CELL 14 — Configure the training run.
# These values are intentionally modest so the example is practical on Colab.

training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",

    # Use mixed precision on a CUDA GPU.
    fp16=torch.cuda.is_available()
)

## Concept: Important hyperparameters

### Learning rate

```text
2e-5
```

Controls how much model weights change during each optimizer update.

### Batch size

```text
8
```

Number of examples processed before one optimizer update.

### Epoch

One complete pass through the training dataset.

### Weight decay

A regularization technique that discourages unnecessarily large weights.

### `fp16`

On compatible NVIDIA GPUs, half-precision training can reduce memory use and improve speed.

In [ ]:
# CELL 15 — Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready.")

## Concept: What does Trainer do?

`Trainer` handles the standard training loop:

```text
batch
 ↓
forward pass
 ↓
loss
 ↓
backward pass
 ↓
gradient calculation
 ↓
optimizer update
 ↓
learning-rate scheduler
```

It also handles evaluation, checkpointing and logging. citeturn0search1turn0search3

In [ ]:
# CELL 16 — Fine-tune BERT

trainer.train()

In [ ]:
# CELL 17 — Evaluate the fine-tuned model

metrics = trainer.evaluate()

print(metrics)

In [ ]:
# CELL 18 — Save the model and tokenizer

save_path = "./bert-imdb-final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved to:", save_path)

In [ ]:
# CELL 19 — Run inference using the fine-tuned model

classifier = pipeline(
    "text-classification",
    model=save_path,
    tokenizer=save_path
)

examples = [
    "This movie was fantastic. I loved the story and acting.",
    "The movie was boring and disappointing. I would not recommend it."
]

for text in examples:
    print("\nText:", text)
    print("Prediction:", classifier(text))

# PART 2 — BERT for Named Entity Recognition (NER)

The original notebook uses WikiAnn and `BertForTokenClassification`.

NER is different from sentiment classification.

Classification:

```text
One sentence → One label
```

NER:

```text
Each token/word → A label
```

Example:

```text
John works at Google in London.

John    → B-PER
works   → O
at      → O
Google  → B-ORG
in      → O
London  → B-LOC
```

WikiAnn uses labels such as `O`, `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`, `B-MISC`, and `I-MISC`.

In [ ]:
# CELL 20 — Load WikiAnn

from datasets import load_dataset

wikiann = load_dataset("wikiann", "en")

print(wikiann)
print(wikiann["train"][0])

In [ ]:
# CELL 21 — Inspect the NER label mapping

ner_features = wikiann["train"].features["ner_tags"]

print(ner_features)
print("\nLabel names:")
print(ner_features.feature.names)

## Concept: B-I-O tagging

For an entity such as:

```text
New York
```

we can represent it as:

```text
New  → B-LOC
York → I-LOC
```

`B` = beginning of an entity  
`I` = inside/continuation of an entity  
`O` = outside an entity

This is why NER has one target label per word rather than one label per sentence.

In [ ]:
# CELL 22 — Use a small subset for a Colab teaching run.

ner_train = (
    wikiann["train"]
    .shuffle(seed=42)
    .select(range(1000))
)

ner_test = (
    wikiann["validation"]
    .shuffle(seed=42)
    .select(range(250))
)

print(len(ner_train), len(ner_test))

In [ ]:
# CELL 23 — Fast tokenizer for word-level alignment

ner_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-cased",
    use_fast=True
)

ner_model_name = "bert-base-cased"

## Important NER concept: subword tokenization

A word can become multiple BERT tokens.

Example:

```text
London
```

may remain one token, while another word may be split:

```text
playing → play + ##ing
```

But the dataset has one NER label for the original word.

Therefore we must **align word-level labels with token-level inputs**.

Special tokens such as `[CLS]`, `[SEP]`, and padding do not receive normal NER labels, so we use `-100` for them. PyTorch's cross-entropy loss ignores positions with `-100`.

In [ ]:
# CELL 24 — Tokenize and align NER labels

def tokenize_and_align_labels(examples):
    tokenized = ner_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256
    )

    all_labels = []

    for batch_index, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=batch_index)

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens / padding
                label_ids.append(-100)

            elif word_id != previous_word_id:
                # First subword of a word gets the original word label
                label_ids.append(labels[word_id])

            else:
                # Subsequent subwords are ignored for this simple setup
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_ner_train = ner_train.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_ner_test = ner_test.map(
    tokenize_and_align_labels,
    batched=True
)

In [ ]:
# CELL 25 — Remove the original text/token columns that are no longer needed

tokenized_ner_train = tokenized_ner_train.remove_columns(
    ["tokens", "ner_tags"]
)

tokenized_ner_test = tokenized_ner_test.remove_columns(
    ["tokens", "ner_tags"]
)

print(tokenized_ner_train.column_names)

In [ ]:
# CELL 26 — NER data collator and model

from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)

label_names = ner_features.feature.names
num_ner_labels = len(label_names)

ner_collator = DataCollatorForTokenClassification(
    tokenizer=ner_tokenizer
)

ner_model = AutoModelForTokenClassification.from_pretrained(
    ner_model_name,
    num_labels=num_ner_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)}
)

print("Number of NER labels:", num_ner_labels)
print(label_names)

## Concept: Why `DataCollatorForTokenClassification`?

For sequence classification, there is one label per example.

For NER, there are many labels per example:

```text
John    → B-PER
works   → O
Google  → B-ORG
```

The collator dynamically pads both:

- token inputs
- token-level labels

and uses `-100` for padded label positions. citeturn0search4

In [ ]:
# CELL 27 — NER metrics

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    # Accuracy over non-ignored token positions.
    mask = labels != -100

    if mask.sum() == 0:
        return {"token_accuracy": 0.0}

    token_accuracy = (
        predictions[mask] == labels[mask]
    ).mean()

    return {
        "token_accuracy": float(token_accuracy)
    }

In [ ]:
# CELL 28 — NER training configuration

ner_training_args = TrainingArguments(
    output_dir="./bert-ner-wikiann",
    num_train_epochs=1,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available()
)

In [ ]:
# CELL 29 — NER Trainer

ner_trainer = Trainer(
    model=ner_model,
    args=ner_training_args,
    train_dataset=tokenized_ner_train,
    eval_dataset=tokenized_ner_test,
    processing_class=ner_tokenizer,
    data_collator=ner_collator,
    compute_metrics=compute_ner_metrics
)

print("NER Trainer ready.")

In [ ]:
# CELL 30 — Fine-tune NER model

ner_trainer.train()

In [ ]:
# CELL 31 — Evaluate NER

ner_metrics = ner_trainer.evaluate()

print(ner_metrics)

## NER prediction

For production NER, the Hugging Face `token-classification` pipeline can convert token-level model predictions into easier-to-read entities.

We will use the fine-tuned model below.

In [ ]:
# CELL 32 — Save and create NER pipeline

ner_save_path = "./bert-ner-wikiann"

ner_trainer.save_model(ner_save_path)
ner_tokenizer.save_pretrained(ner_save_path)

ner_pipeline = pipeline(
    "token-classification",
    model=ner_save_path,
    tokenizer=ner_save_path,
    aggregation_strategy="simple"
)

text = "Apple was founded by Steve Jobs in California."

entities = ner_pipeline(text)

print(entities)

# PART 3 — Extractive Question Answering

The original notebook uses SQuAD and `BertForQuestionAnswering`.

QA is different from both previous tasks.

Input:

```text
Question + Context
```

Example:

```text
Question:
What is the capital of France?

Context:
France is a country in Europe. Paris is the capital of France.
```

Expected answer:

```text
Paris
```

BERT QA predicts two positions:

```text
start position
end position
```

If the answer spans tokens 7 through 7:

```text
start = 7
end   = 7
```

For a multi-token answer:

```text
New York
```

the model predicts:

```text
start → New
end   → York
```

In [ ]:
# CELL 33 — Load SQuAD

squad = load_dataset("squad")

print(squad)
print(squad["train"][0])

## Concept: Character positions vs token positions

SQuAD provides the answer location using character offsets in the context:

```python
answer_start
```

BERT needs token positions.

Therefore QA preprocessing must convert:

```text
character offset
        ↓
token start/end positions
```

This is why QA preprocessing is more complicated than ordinary classification.

In [ ]:
# CELL 34 — Create a small SQuAD subset for demonstration

qa_train = (
    squad["train"]
    .shuffle(seed=42)
    .select(range(1000))
)

qa_validation = (
    squad["validation"]
    .shuffle(seed=42)
    .select(range(250))
)

qa_model_name = "bert-base-uncased"

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name,
    use_fast=True
)

print("Train:", len(qa_train))
print("Validation:", len(qa_validation))

In [ ]:
# CELL 35 — QA preprocessing
# We use offset mappings to convert answer character positions
# into token start/end positions.

max_length = 384

def prepare_qa_features(examples):
    questions = [
        q.strip() for q in examples["question"]
    ]

    tokenized = qa_tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_index = sample_mapping[i]

        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = tokenized.sequence_ids(i)

        # Find the context token range.
        context_start = 0

        while (
            context_start < len(sequence_ids)
            and sequence_ids[context_start] != 1
        ):
            context_start += 1

        context_end = len(sequence_ids) - 1

        while (
            context_end >= 0
            and sequence_ids[context_end] != 1
        ):
            context_end -= 1

        # If the answer is not completely inside this feature,
        # mark the feature as having no answer.
        if (
            context_start > context_end
            or offsets[context_start][0] > start_char
            or offsets[context_end][1] < end_char
        ):
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find token containing the start character.
        token_start = context_start

        while (
            token_start <= context_end
            and offsets[token_start][0] <= start_char
        ):
            token_start += 1

        start_positions.append(token_start - 1)

        # Find token containing the end character.
        token_end = context_end

        while (
            token_end >= context_start
            and offsets[token_end][1] >= end_char
        ):
            token_end -= 1

        end_positions.append(token_end + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized


tokenized_qa_train = qa_train.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_train.column_names
)

tokenized_qa_validation = qa_validation.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_validation.column_names
)

print(tokenized_qa_train.column_names)

## Important QA note

QA examples can create **multiple tokenized features** because a long context may be split into overlapping chunks.

That is why the preprocessing uses:

```text
max_length
stride
return_overflowing_tokens=True
```

This is different from simple sentiment classification.

For a production QA evaluation pipeline, answer extraction and exact-match/F1 evaluation need additional post-processing from token predictions back to the original context. This notebook keeps the training concept focused rather than hiding that complexity.

In [ ]:
# CELL 36 — QA model

from transformers import (
    AutoModelForQuestionAnswering,
    default_data_collator
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)

print("QA model loaded.")

In [ ]:
# CELL 37 — QA Trainer configuration

qa_training_args = TrainingArguments(
    output_dir="./bert-squad",
    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=3e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available()
)

qa_trainer = Trainer(
    model=qa_model,
    args=qa_training_args,
    train_dataset=tokenized_qa_train,
    eval_dataset=tokenized_qa_validation,
    processing_class=qa_tokenizer,
    data_collator=default_data_collator
)

print("QA Trainer ready.")

In [ ]:
# CELL 38 — Fine-tune QA model

qa_trainer.train()

In [ ]:
# CELL 39 — Save QA model

qa_save_path = "./bert-squad-final"

qa_trainer.save_model(qa_save_path)
qa_tokenizer.save_pretrained(qa_save_path)

print("Saved:", qa_save_path)

In [ ]:
# CELL 40 — Simple QA inference

qa_pipeline = pipeline(
    "question-answering",
    model=qa_save_path,
    tokenizer=qa_save_path
)

question = "What is the capital of France?"

context = (
    "France is a country in Europe. "
    "Paris is the capital and largest city of France. "
    "It is known for the Eiffel Tower."
)

result = qa_pipeline(
    question=question,
    context=context
)

print(result)

# PART 4 — Hugging Face Hub

The old notebook used:

```python
from huggingface_hub import notebook_login
notebook_login()
```

That notebook login helper may still be useful, but the current CLI is:

```bash
hf auth login
```

Do **not** use the old:

```bash
huggingface-cli login
```

if your environment reports that command as missing.

In [ ]:
# CELL 41 — Optional: authenticate with Hugging Face

# Run this only if you want to upload the model.
# You will be asked for your Hugging Face access token.

!hf auth login

In [ ]:
# CELL 42 — Optional: push the fine-tuned IMDb model to the Hub

# Replace this with YOUR Hugging Face username and a repository name
# that you want to create.

# trainer.push_to_hub("my-bert-imdb")

# Final mental model

You should now be able to explain the three fine-tuning tasks like this:

## 1. Sequence Classification

```text
Text
 ↓
Tokenizer
 ↓
BERT
 ↓
Classification Head
 ↓
One label
```

Example:

```text
"This movie is amazing"
          ↓
       POSITIVE
```

## 2. Token Classification / NER

```text
Text
 ↓
Tokenizer + word/token alignment
 ↓
BERT
 ↓
Token Classification Head
 ↓
One label per token
```

Example:

```text
John   → B-PER
works  → O
Google → B-ORG
```

## 3. Question Answering

```text
Question + Context
        ↓
     Tokenizer
        ↓
       BERT
        ↓
Start-position head + End-position head
        ↓
Answer span
```

Example:

```text
Question: What is the capital of France?
Context:  Paris is the capital of France.

Answer: Paris
```

### The common fine-tuning pattern

```text
                PRETRAINED BERT
                      ↓
              Task-specific head
                      ↓
             Labeled training data
                      ↓
                 Loss function
                      ↓
              Backpropagation
                      ↓
              Updated weights
                      ↓
               Fine-tuned model
```

The Hugging Face `Trainer` abstracts the standard training loop while still exposing the important training configuration. citeturn0search1turn0search3

# Practice exercise

After you understand the IMDb example, replace IMDb with your own dataset.

For example:

```text
text,label
"Unable to login",0
"Payment deducted twice",1
"Database connection timeout",2
"Password reset failed",0
```

Then change:

```text
0 → Authentication
1 → Billing
2 → Technical
```

This turns the tutorial into an **enterprise support-ticket classifier**, which is much closer to a real-world AI/ML project.

The preprocessing pattern stays almost identical:

```text
CSV
 ↓
Hugging Face Dataset
 ↓
Tokenizer
 ↓
BERT
 ↓
Classification Head
 ↓
Fine-tuning
 ↓
Evaluation
 ↓
Inference
```